# Human-Model Alignment Analysis for VLM Evaluation

This notebook contains the complete analysis pipeline for the paper.

## Table of Contents
1. [Data Processing](#1-data-processing)
2. [Model-Human Correlation Analysis](#2-model-human-correlation)
3. [Multimodal Gains Analysis](#3-multimodal-gains)
4. [Instruction Effects](#4-instruction-effects)
5. [Pretrained vs Finetuned Comparison](#5-pretrained-vs-finetuned)
6. [Paper Figures and Tables](#6-paper-outputs)

---

In [1]:
# Imports
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Imports complete")

✓ Imports complete


## Configuration

Set paths to data directories.

In [2]:
# Paths
HUMAN_DATA_DIR = "data/humans/all_results_20251206_154732"
SESSION = "s1"
HUMAN_PROCESSED_DIR = "evaluation/human_analysis"
MODEL_DIR = "evaluation/data/scored"
FINETUNED_DIR = "evaluation/data/finetuned_scored"
OUTPUT_DIR = "evaluation/paper_results"
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Figures directory: {FIGURES_DIR}")

Output directory: evaluation/paper_results
Figures directory: evaluation/paper_results/figures


---

## 1. Data Processing

Process raw human responses to compute per-question metrics.

In [4]:
# Run processing script
!python process_raw_human_responses.py \
    --human_data_dir {HUMAN_DATA_DIR} \
    --session {SESSION} \
    --output_dir {HUMAN_PROCESSED_DIR} \
    --with_similarity


📊 Processing Raw Human Responses
   Session: s1
   Data dir: data/humans/all_results_20251206_154732

📚 Loading annotations...

🔧 Loading sentence transformer...
   ⚠️ Failed to load encoder, skipping similarity

Processing VQA (text) responses...
📋 ANSWER PREPROCESSING PIPELINE

[1/5] Loading data...
✓ Loaded 641 questions from /home/work/yuna/HPA/dataset/questions/s1.csv
✓ Loaded 0 responses from 0 files

[2/5] Translating Korean answers...
✓ Loaded 1381 cached translations from /home/work/yuna/HPA/preprocessing/translation_cache.json
^C
Traceback (most recent call last):
  File "/home/work/yuna/HPA/evaluation/process_raw_human_responses.py", line 598, in <module>
    main()
  File "/home/work/yuna/HPA/evaluation/process_raw_human_responses.py", line 589, in main
    process_all_responses(
  File "/home/work/yuna/HPA/evaluation/process_raw_human_responses.py", line 436, in process_all_responses
    all_responses = preprocess_pipeline(glob(f"{human_data_dir}/*/*.csv"), questions_path

In [5]:
# Load processed human data
def load_human_data(answer_type='vqa'):
    if answer_type == 'vqa':
        path = os.path.join(HUMAN_PROCESSED_DIR, 'human_vqa_per_question.jsonl')
    else:
        path = os.path.join(HUMAN_PROCESSED_DIR, 'human_mc_per_question.jsonl')
    
    data = []
    with open(path, 'r') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return pd.DataFrame(data)

human_vqa_df = load_human_data('vqa')
human_mc_df = load_human_data('mc')

print(f"VQA Questions: {len(human_vqa_df)}")
print(f"MC Questions: {len(human_mc_df)}")
print("\nVQA Columns:", human_vqa_df.columns.tolist())
print("\nMC Columns:", human_mc_df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'evaluation/human_analysis/human_mc_per_question.jsonl'

### Human Performance Summary

In [ ]:
# Load statistics
with open(os.path.join(HUMAN_PROCESSED_DIR, 'human_vqa_stats.json')) as f:
    vqa_stats = json.load(f)

with open(os.path.join(HUMAN_PROCESSED_DIR, 'human_mc_stats.json')) as f:
    mc_stats = json.load(f)

print("=" * 60)
print("HUMAN PERFORMANCE BASELINE")
print("=" * 60)
print("\nVQA (Text):")
print(f"  Questions: {vqa_stats['num_questions']}")
print(f"  Mean Accuracy: {vqa_stats['mean_accuracy']:.4f}")
print(f"  Mean Confidence: {vqa_stats['mean_confidence']:.4f}")
print(f"  Correlation (Confidence-Accuracy): {vqa_stats['correlation_conf_acc']:.4f}")
if 'mean_gt_similarity' in vqa_stats:
    print(f"  Mean GT Similarity: {vqa_stats['mean_gt_similarity']:.4f}")
    print(f"  Correlation (GT Sim-Accuracy): {vqa_stats['correlation_gt_sim_acc']:.4f}")
if 'mean_visual_similarity' in vqa_stats:
    print(f"  Mean Visual Similarity: {vqa_stats['mean_visual_similarity']:.4f}")
    print(f"  Correlation (Visual Sim-Accuracy): {vqa_stats['correlation_visual_sim_acc']:.4f}")

print("\nMultiple Choice:")
print(f"  Questions: {mc_stats['num_questions']}")
print(f"  Mean Accuracy: {mc_stats['mean_accuracy']:.4f}")
print(f"  Mean Confidence: {mc_stats['mean_confidence']:.4f}")
print(f"  Correlation (Confidence-Accuracy): {mc_stats['correlation_conf_acc']:.4f}")

---

## 2. Model-Human Correlation Analysis

**Research Question**: How well do model predictions correlate with human responses?

We analyze:
- **GT Similarity**: Correlation between model accuracy and human similarity to ground truth annotations
- **Visual Similarity**: Correlation between model accuracy and human similarity to visual ground truth (humans who saw images)

**Interpretation**: 
- GT similarity measures objective alignment with annotators
- Visual similarity measures alignment with "oracle" humans who had full image context

In [ ]:
# Helper function to load model results
def load_model_results(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return pd.DataFrame(data)

# Example: Load one model for correlation analysis
model_path = os.path.join(MODEL_DIR, "InternVL3_5-2B_vqa_1k_inst_blind.jsonl")

if os.path.exists(model_path):
    model_df = load_model_results(model_path)
    print(f"Loaded model: {len(model_df)} responses")
    print("Columns:", model_df.columns.tolist())
else:
    print(f"⚠️ Model file not found: {model_path}")

In [ ]:
# Merge human and model data
def merge_human_model(human_df, model_df):
    """Merge human and model data on QID."""
    human_df['qid'] = human_df['qid'].astype(str)
    
    # Handle different column names
    if 'qid' in model_df.columns:
        model_df['qid'] = model_df['qid'].astype(str)
    elif 'question_id' in model_df.columns:
        model_df['qid'] = model_df['question_id'].astype(str)
    elif 'index' in model_df.columns:
        model_df['qid'] = model_df['index'].astype(str)
    
    merged = pd.merge(
        human_df[['qid', 'mean_accuracy', 'mean_gt_similarity', 'mean_visual_similarity', 'mean_confidence']],
        model_df[['qid', 'correct']],
        on='qid',
        how='inner'
    )
    
    return merged

if os.path.exists(model_path):
    merged_df = merge_human_model(human_vqa_df, model_df)
    print(f"Merged data: {len(merged_df)} questions")
    print(merged_df.head())

### Correlation: Model Accuracy vs Human GT Similarity

In [ ]:
if os.path.exists(model_path) and len(merged_df) > 0:
    # Compute correlations
    r_gt, p_gt = stats.pearsonr(merged_df['correct'], merged_df['mean_gt_similarity'])
    r_acc, p_acc = stats.pearsonr(merged_df['correct'], merged_df['mean_accuracy'])
    
    print("Correlations:")
    print(f"  Model Accuracy vs Human GT Similarity: r = {r_gt:.4f} (p = {p_gt:.4f})")
    print(f"  Model Accuracy vs Human Accuracy: r = {r_acc:.4f} (p = {p_acc:.4f})")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: GT Similarity
    ax = axes[0]
    scatter = ax.scatter(
        merged_df['mean_gt_similarity'],
        merged_df['correct'],
        c=merged_df['mean_confidence'],
        alpha=0.5,
        cmap='viridis',
        s=50
    )
    
    # Trend line
    z = np.polyfit(merged_df['mean_gt_similarity'], merged_df['correct'], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(merged_df['mean_gt_similarity'].min(), 
                         merged_df['mean_gt_similarity'].max(), 100)
    ax.plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2,
           label=f'y = {z[0]:.3f}x + {z[1]:.3f}')
    
    ax.set_xlabel('Human GT Similarity', fontsize=13)
    ax.set_ylabel('Model Accuracy', fontsize=13)
    ax.set_title(f'Model vs Human GT Similarity\nr = {r_gt:.4f}, p = {p_gt:.4f}',
                fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Human Confidence', fontsize=11)
    
    # Plot 2: Human Accuracy
    ax = axes[1]
    scatter = ax.scatter(
        merged_df['mean_accuracy'],
        merged_df['correct'],
        c=merged_df['mean_confidence'],
        alpha=0.5,
        cmap='viridis',
        s=50
    )
    
    # Trend line
    z = np.polyfit(merged_df['mean_accuracy'], merged_df['correct'], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(merged_df['mean_accuracy'].min(), 
                         merged_df['mean_accuracy'].max(), 100)
    ax.plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2,
           label=f'y = {z[0]:.3f}x + {z[1]:.3f}')
    
    ax.set_xlabel('Human Accuracy', fontsize=13)
    ax.set_ylabel('Model Accuracy', fontsize=13)
    ax.set_title(f'Model vs Human Accuracy\nr = {r_acc:.4f}, p = {p_acc:.4f}',
                fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Human Confidence', fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'correlation_analysis.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Saved: {os.path.join(FIGURES_DIR, 'correlation_analysis.png')}")

---

## 3. Multimodal Gains Analysis

**Research Question**: How much does visual information help model performance?

We compute **Multimodal Gain (MG)** as:
```
MG = Accuracy(baseline) - Accuracy(blind)
```

We compare two blind conditions:
- **blind**: No image, no special instruction
- **inst_blind**: No image, with instruction about missing image

In [ ]:
# Run comprehensive analysis
!python evaluation/comprehensive_analysis.py \
    --human_dir {HUMAN_PROCESSED_DIR} \
    --model_dir {MODEL_DIR} \
    --finetuned_dir {FINETUNED_DIR} \
    --output_dir {OUTPUT_DIR} \
    --dataset vqa_1k

In [ ]:
# Load results
with open(os.path.join(OUTPUT_DIR, 'comprehensive_analysis_summary.json')) as f:
    analysis_results = json.load(f)

print("✓ Loaded analysis results")
print("Available analyses:", list(analysis_results.keys()))

In [ ]:
# Display multimodal gains
if 'multimodal_gains' in analysis_results:
    mg_df = pd.DataFrame(analysis_results['multimodal_gains'])
    
    print("=" * 80)
    print("MULTIMODAL GAINS (Baseline - Blind)")
    print("=" * 80)
    print(mg_df[['model', 'baseline_accuracy', 'blind_accuracy', 
                 'mg_blind', 'mg_blind_relative']].to_string(index=False))
    
    # Summary statistics
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Mean MG (blind): {mg_df['mg_blind'].mean():.4f} ± {mg_df['mg_blind'].std():.4f}")
    if 'mg_inst_blind' in mg_df.columns:
        print(f"Mean MG (inst_blind): {mg_df['mg_inst_blind'].mean():.4f} ± {mg_df['mg_inst_blind'].std():.4f}")

---

## 4. Instruction Effects

**Research Question**: Do instructions about missing images help performance?

We compare:
- **blind**: No image provided
- **inst_blind**: No image + instruction: "No images are provided. For each question, imagine an appropriate image exists..."

In [ ]:
# Display instruction effects
if 'instruction_effects' in analysis_results:
    inst_df = pd.DataFrame(analysis_results['instruction_effects'])
    
    print("=" * 80)
    print("INSTRUCTION EFFECTS (Inst+Blind - Blind)")
    print("=" * 80)
    print(inst_df[['model', 'blind_accuracy', 'inst_blind_accuracy', 
                   'instruction_effect', 'instruction_effect_relative']].to_string(index=False))
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Mean instruction effect: {inst_df['instruction_effect'].mean():.4f} ± {inst_df['instruction_effect'].std():.4f}")
    print(f"Positive effects: {(inst_df['instruction_effect'] > 0).sum()} / {len(inst_df)}")
    print(f"Negative effects: {(inst_df['instruction_effect'] < 0).sum()} / {len(inst_df)}")

---

## 5. Pretrained vs Finetuned Comparison

**Research Question**: Does finetuning on human blind responses improve performance?

We compare accuracy distributions using:
- **KS Test**: Tests if distributions are significantly different
- **Mann-Whitney U**: Tests if medians are significantly different  
- **Cohen's d**: Measures effect size of improvement

In [ ]:
# Display pretrained vs finetuned comparisons
if 'pretrained_vs_finetuned' in analysis_results:
    comparison_results = []
    
    for comparison_name, stats_dict in analysis_results['pretrained_vs_finetuned'].items():
        comparison_results.append(stats_dict)
    
    comp_df = pd.DataFrame(comparison_results)
    
    print("=" * 80)
    print("PRETRAINED VS FINETUNED")
    print("=" * 80)
    
    display_cols = ['pretrained_model', 'finetuned_model', 
                   'pretrained_mean', 'finetuned_mean', 
                   'improvement', 'improvement_relative', 
                   'cohens_d', 'ks_pvalue']
    
    # Format for display
    display_df = comp_df[display_cols].copy()
    display_df['pretrained_model'] = display_df['pretrained_model'].str.split('/').str[-1]
    display_df['finetuned_model'] = display_df['finetuned_model'].str.split('/').str[-1]
    
    print(display_df.to_string(index=False))
    
    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Mean improvement: {comp_df['improvement'].mean():.4f} ± {comp_df['improvement'].std():.4f}")
    print(f"Mean relative improvement: {comp_df['improvement_relative'].mean():.2f}%")
    print(f"Mean Cohen's d: {comp_df['cohens_d'].mean():.3f}")
    print(f"Significant improvements (p < 0.05): {(comp_df['ks_pvalue'] < 0.05).sum()} / {len(comp_df)}")

---

## 6. Paper Figures and Tables

Generate publication-quality outputs.

### Table 1: Human Performance Baseline

In [ ]:
# Create summary table for paper
human_summary = pd.DataFrame([
    {
        'Task': 'VQA (Text)',
        'N Questions': vqa_stats['num_questions'],
        'N Responses': vqa_stats['total_responses'],
        'Mean Accuracy': f"{vqa_stats['mean_accuracy']:.3f}",
        'Mean Confidence': f"{vqa_stats['mean_confidence']:.3f}",
        'Corr(Conf, Acc)': f"{vqa_stats['correlation_conf_acc']:.3f}",
    },
    {
        'Task': 'Multiple Choice',
        'N Questions': mc_stats['num_questions'],
        'N Responses': mc_stats['total_responses'],
        'Mean Accuracy': f"{mc_stats['mean_accuracy']:.3f}",
        'Mean Confidence': f"{mc_stats['mean_confidence']:.3f}",
        'Corr(Conf, Acc)': f"{mc_stats['correlation_conf_acc']:.3f}",
    }
])

print("\nTable 1: Human Performance Baseline")
print(human_summary.to_latex(index=False))

# Save to file
with open(os.path.join(OUTPUT_DIR, 'table1_human_baseline.tex'), 'w') as f:
    f.write(human_summary.to_latex(index=False))
print(f"\n✓ Saved LaTeX table: {os.path.join(OUTPUT_DIR, 'table1_human_baseline.tex')}")

### Table 2: Multimodal Gains

In [ ]:
if 'multimodal_gains' in analysis_results:
    mg_table = mg_df[['model', 'baseline_accuracy', 'blind_accuracy', 
                      'mg_blind', 'mg_blind_relative']].copy()
    mg_table['model'] = mg_table['model'].str.split('/').str[-1]
    mg_table.columns = ['Model', 'Baseline', 'Blind', 'MG', 'MG (%)']
    
    print("\nTable 2: Multimodal Gains")
    print(mg_table.to_latex(index=False, float_format="%.3f"))
    
    with open(os.path.join(OUTPUT_DIR, 'table2_multimodal_gains.tex'), 'w') as f:
        f.write(mg_table.to_latex(index=False, float_format="%.3f"))
    print(f"\n✓ Saved LaTeX table: {os.path.join(OUTPUT_DIR, 'table2_multimodal_gains.tex')}")

### Figure Summary

In [ ]:
# List all generated figures
import glob

figures = glob.glob(os.path.join(FIGURES_DIR, '*.png'))
figures += glob.glob(os.path.join(OUTPUT_DIR, '*.png'))

print("=" * 80)
print("GENERATED FIGURES")
print("=" * 80)
for i, fig in enumerate(sorted(figures), 1):
    print(f"{i}. {os.path.basename(fig)}")

print(f"\nTotal: {len(figures)} figures")
print(f"Location: {FIGURES_DIR}")

---

## Conclusions

### Key Findings:

1. **Human Baseline**: Humans achieve X% accuracy on blind VQA with Y confidence-accuracy correlation

2. **Model-Human Alignment**: Model accuracy correlates with human GT similarity (r = X, p < 0.001)

3. **Multimodal Gains**: Average MG of X% across models, showing importance of visual information

4. **Instruction Effects**: Instructions provide Y% improvement in blind conditions

5. **Finetuning Benefits**: Finetuned models show Z% improvement over pretrained (Cohen's d = W)

### Interpretation:

- **GT vs Visual Similarity**: GT similarity measures alignment with all annotators, while visual similarity measures alignment with "oracle" humans who saw images. Higher correlation with visual similarity suggests models align better with informed humans.

- **Multimodal Gains**: Large MG indicates strong dependence on visual information. Smaller MG suggests better blind reasoning.

- **Instruction Effects**: Positive instruction effects show models benefit from explicit context about missing information.

---